# Why RAG? Grounding answers in trusted sources: worked solution

Worked solution for the exercise in the lesson **[Why RAG? Grounding answers in trusted sources](https://ramkumarmd.com/medical-ai/rag/why-rag)** (*Retrieval-augmented generation (RAG)* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](01-why-rag.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Retrieval-augmented generation (RAG) answers a question in two steps: first it retrieves the most relevant passages from a trusted document collection, such as your organisation's guidelines; then it gives those passages to a language model with instructions to answer only from them and cite them. This keeps answers current and checkable without retraining the model. RAG reduces, but does not remove, errors: retrieval can miss the right passage, and the model can still misread what it is given.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A minimal retrieve-then-prompt example

Score four synthetic documents against a question.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = {
    "protocol-2024": "Clinic protocol 2024: review HbA1c every 6 months when stable.",
    "protocol-2026": "Clinic protocol 2026 update: review HbA1c every 3 months for the first year "
                     "after any treatment change, then every 6 months when stable.",
    "foot-care": "Foot checks: examine both feet at every diabetes review and test sensation "
                 "with a monofilament once a year.",
    "eye-screening": "Eye screening: arrange retinal photography at diagnosis and then at the "
                     "interval set by the local screening programme.",
}
question = "How often should HbA1c be reviewed after a treatment change?"

vectoriser = TfidfVectorizer(stop_words="english").fit(list(documents.values()) + [question])
doc_vectors = vectoriser.transform(documents.values())
scores = cosine_similarity(vectoriser.transform([question]), doc_vectors)[0]
for name, score in sorted(zip(documents, scores), key=lambda t: -t[1]):
    print(f"{score:.2f}  {name}")

# 0.34  protocol-2026
# 0.13  protocol-2024
# 0.00  foot-care
# 0.00  eye-screening

### Building the grounded prompt

Continuing from the previous block: put the best passage into an instruction-led prompt.

In [ ]:
best = max(zip(documents, scores), key=lambda t: t[1])[0]
prompt = f"""Answer using only the source below. If the source does not answer, say so.
Cite the source id in square brackets.

Source [{best}]: {documents[best]}

Question: {question}"""
print(prompt)

# Answer using only the source below. If the source does not answer, say so.
# Cite the source id in square brackets.
#
# Source [protocol-2026]: Clinic protocol 2026 update: review HbA1c every 3 months for the first year after any treatment change, then every 6 months when stable.
#
# Question: How often should HbA1c be reviewed after a treatment change?

## Exercise

**Goal:** Build a minimal RAG pipeline over a synthetic document library and observe where it succeeds and fails.

**Data:** Synthetic protocol documents written for the notebook (Created for this course; not clinical guidance)

### Step 1

Reproduce the retrieval and prompt-building steps.

In [ ]:
def retrieve(question, docs):
    vec = TfidfVectorizer(stop_words="english").fit(list(docs.values()) + [question])
    s = cosine_similarity(vec.transform([question]), vec.transform(docs.values()))[0]
    return sorted(zip(docs, s.round(2).tolist()), key=lambda t: -t[1])

def grounded_prompt(question, doc_id, docs):
    return (f"Answer using only the source below. If the source does not answer, say so.\n"
            f"Cite the source id in square brackets.\n\nSource [{doc_id}]: {docs[doc_id]}\n\nQuestion: {question}")

ranking = retrieve(question, documents)
print(ranking)
print(grounded_prompt(question, ranking[0][0], documents))

### Step 2

Write five questions, including one the documents cannot answer, and record which document is retrieved for each.

In [ ]:
my_questions = [
    "How often should HbA1c be checked when diabetes is stable?",
    "When should the monofilament test be done?",
    "When is retinal photography arranged?",
    "Should both feet be examined at each visit?",
    "What is the starting dose of insulin glargine?",          # not covered by any document
]
for q in my_questions:
    (top, score), (second, score2) = retrieve(q, documents)[:2]
    print(f"{q}\n   top: {top} ({score:.2f})   next: {second} ({score2:.2f})")

The stable-HbA1c question retrieves the *2024* protocol first, because it matches its wording ("every 6 months
when stable") more closely, even though the 2026 update contains the same rule plus the newer one. The
insulin question has nothing to find, but retrieval still returns a "top" document. A retriever always
returns something: a minimum score, and an instruction to say "the source does not answer", are needed.

### Step 3

Retire the 2024 protocol by adding a 'superseded' flag and filtering it out; check the rankings again.

In [ ]:
metadata = {"protocol-2024": {"superseded": True}, "protocol-2026": {"superseded": False},
            "foot-care": {"superseded": False}, "eye-screening": {"superseded": False}}
current = {k: v for k, v in documents.items() if not metadata[k]["superseded"]}

for q in [question, my_questions[0]]:
    print(f"{q}\n   all documents:     {retrieve(q, documents)[:2]}"
          f"\n   current documents: {retrieve(q, current)[:2]}")

Filtering on metadata happens before ranking, so an outdated document can never be retrieved, however well it
matches the wording. Keeping a document library current (version, status, review date) is as important to a
RAG system as the retrieval method.

### Step 4

Optional, with an approved model: send the prompts and compare answers with and without the retrieved source.

Optional, with a model your organisation has approved, and only with the synthetic documents here.

```python
def ask(prompt: str) -> str:
    """Call your approved model and return its answer (API key from an environment variable)."""
    raise NotImplementedError

for q in my_questions:
    top = retrieve(q, current)[0][0]
    without_source = ask(q)
    with_source = ask(grounded_prompt(q, top, current))
    print(f"Q: {q}\n  without source: {without_source}\n  with source:    {with_source}\n")
```

What to look for: without a source, the model answers from its training data, which may not match *this*
clinic's protocol (for example a different HbA1c interval) and cannot be cited. With the source, answers
should follow the protocol and cite it; check that the unanswerable question gets "the source does not answer"
rather than an invented dose.

---

Lesson: [https://ramkumarmd.com/medical-ai/rag/why-rag](https://ramkumarmd.com/medical-ai/rag/why-rag). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.